In [503]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

In [504]:
kidney = pd.read_csv('../Data/trimmed_kidney_disease_random_1000_patients_version2.csv.gz',
                    compression = 'gzip')
kidney.head()

,Unnamed: 0,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,0,268992,10015860,22413744.0,1702357,50853,NaN,2191-01-27 06:25:00,2191-01-28 13:25:00,___,11.0,ng/mL,30.0,60.0,abnormal,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
1,2,1952705,10132512,NaN,72063012,50853,P30YWU,2178-02-12 08:16:00,2178-02-13 09:42:00,___,32.0,ng/mL,30.0,60.0,NaN,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
2,3,1952726,10132512,NaN,77832825,50853,P30YWU,2178-05-16 07:55:00,2178-05-18 08:13:00,___,42.0,ng/mL,30.0,60.0,NaN,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
3,4,1952831,10132512,NaN,30613615,50853,P84CA4,2179-03-09 14:36:00,2179-03-10 13:31:00,___,33.0,ng/mL,30.0,60.0,NaN,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
4,5,1953127,10132512,NaN,98177482,50853,P84CA4,2180-08-24 15:35:00,2180-08-25 11:45:00,___,41.0,ng/mL,30.0,60.0,NaN,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...


In [505]:
non_kidney = pd.read_csv('../Data/trimmed_normal_kidney_patients.csv.gz',
                        compression = 'gzip')
non_kidney.head()

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,656869,10041351,28202624.0,87960306,50853,NaN,2117-09-24 06:24:00,2117-09-26 14:57:00,20,20.0,ng/mL,30.0,60.0,abnormal,ROUTINE,NaN
1,206675,10012703,NaN,65289690,50853,P134DE,2184-02-27 13:45:00,2184-02-28 10:30:00,___,4.0,ng/mL,30.0,60.0,abnormal,ROUTINE,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
2,206771,10012703,NaN,45239687,50853,P134DE,2184-06-05 10:30:00,2184-06-08 10:45:00,___,24.0,ng/mL,30.0,60.0,abnormal,STAT,NEW ASSAY IN USE ___: DETECTS D2 AND D3 25-OH ...
3,680714,10043423,NaN,40263589,50853,P941BA,2140-07-04 15:15:00,2140-07-06 15:11:00,8,8.0,ng/mL,30.0,60.0,abnormal,ROUTINE,NaN
4,255932,10014790,NaN,89220377,50853,P61084,2171-10-26 14:35:00,2171-10-26 19:17:00,___,46.0,ng/mL,30.0,60.0,NaN,ROUTINE,"Effective ___, values will be reported from th..."


In [506]:
# First, find which dates have all 3 itemids for each subject
n_labs = 8
complete_dates = kidney.groupby(['subject_id', 'charttime'])['itemid'].apply(
    lambda x: len(set(x) & {51006, 50882, 50912, 50862, 50971, 50902, 50893, 51222}) == n_labs
).reset_index()
complete_dates = complete_dates[complete_dates['itemid']].drop('itemid', axis=1)

# Get the latest complete date for each subject
latest_dates = complete_dates.groupby('subject_id')['charttime'].max().reset_index()

# Merge back to get the actual lab data
latest_complete_labs = kidney.merge(latest_dates, on=['subject_id', 'charttime'])

In [507]:
# First, find which dates have all 3 itemids for each subject
Complete_Dates = non_kidney.groupby(['subject_id', 'charttime'])['itemid'].apply(
    lambda x: len(set(x) & {51006, 50882, 50912, 50862, 50971, 50902, 50893, 51222}) == n_labs
).reset_index()
Complete_Dates = Complete_Dates[Complete_Dates['itemid']].drop('itemid', axis=1)

# Get all unique subject_ids from original dataset
all_subjects = set(non_kidney['subject_id'].unique())

# Get subjects that have at least one complete date
subjects_with_complete_dates = set(Complete_Dates['subject_id'].unique())

# Find subjects that DON'T have all three itemids on any single day
subjects_without_complete_dates = all_subjects - subjects_with_complete_dates



# Get the latest complete date for each subject (only for those who have complete dates)
Latest_Dates = Complete_Dates.groupby('subject_id')['charttime'].max().reset_index()


# Merge back to get the actual lab data (only for subjects with complete dates)

Latest_Complete_Labs = non_kidney.merge(Latest_Dates, on=['subject_id', 'charttime'])

In [508]:
filtered_latest_complete_labs = latest_complete_labs[latest_complete_labs['itemid'].isin([51006, 50882, 50912, 50862, 50971, 50902, 50893, 51222])]
Filtered_Latest_Complete_Labs = Latest_Complete_Labs[Latest_Complete_Labs['itemid'].isin([51006, 50882, 50912, 50862, 50971, 50902, 50893, 51222])]

In [509]:
# Count NaN values in 'valuenum' column
kid_nan_count = filtered_latest_complete_labs['valuenum'].isna().sum()
print(kid_nan_count)
non_kid_nan_count= Filtered_Latest_Complete_Labs['valuenum'].isna().sum()
print(non_kid_nan_count)

2
3


In [510]:
# Step 1: Find all subject_ids that have NaN in 'value' column
problematic_subjects = filtered_latest_complete_labs[filtered_latest_complete_labs['valuenum'].isna()]['subject_id'].unique()
print(f"Subject IDs with NaN in 'value': {problematic_subjects}")

# Step 2: Remove all rows with those subject_ids
filtered_latest_complete_labs = filtered_latest_complete_labs[~filtered_latest_complete_labs['subject_id'].isin(problematic_subjects)]

Subject IDs with NaN in 'value': [10174039 10800948]


In [511]:
# Step 1: Find all subject_ids that have NaN in 'value' column
problematic_subjects = Filtered_Latest_Complete_Labs[Filtered_Latest_Complete_Labs['valuenum'].isna()]['subject_id'].unique()
print(f"Subject IDs with NaN in 'value': {problematic_subjects}")

# Step 2: Remove all rows with those subject_ids
Filtered_Latest_Complete_Labs = Filtered_Latest_Complete_Labs[~Filtered_Latest_Complete_Labs['subject_id'].isin(problematic_subjects)]

Subject IDs with NaN in 'value': [10006441 10048262 10056838]


In [512]:
# Use pivot_table with aggregation to handle duplicates
new_df1 = filtered_latest_complete_labs.pivot_table(index='subject_id', columns='itemid', values='valuenum', aggfunc='mean').reset_index()

# Remove the 'itemid' column name from the header
new_df1.columns.name = None

new_df2 = Filtered_Latest_Complete_Labs.pivot_table(index='subject_id', columns='itemid', values='valuenum',aggfunc='mean').reset_index()
new_df2.columns.name = None

In [513]:
# Add the KD column with value 1 for all subjects
new_df1['Class'] = 1

#Add the KD column with value 0 for all subjects
new_df2['Class']=0
new_df2.head()

,subject_id,50862,50882,50893,50902,50912,50971,51006,51222,Class
0,10000032,3.5,24.0,8.8,93.0,0.4,5.4,29.0,11.6,0
1,10000084,3.9,24.0,9.6,104.0,0.8,4.1,18.0,13.0,0
2,10000117,5.0,24.0,9.7,98.0,1.0,3.8,10.0,14.7,0
3,10000826,3.8,28.0,9.2,102.0,0.6,4.5,5.0,11.4,0
4,10000935,1.6,20.0,8.4,111.0,0.3,3.3,8.0,8.2,0


In [514]:
print(new_df1.shape)
print(new_df2.shape)

(572, 10)
(382, 10)


In [515]:
new_df1 = new_df1.sample(new_df2.shape[0])
combined_data = pd.concat([new_df1, new_df2], 
                ignore_index = True)
data_shuffled = combined_data.sample(frac = 1).reset_index(drop = True)
column_names = data_shuffled.columns
data = data_shuffled.rename(columns = {50862: 'albumin',
                             50882: 'bicarbonate',
                             50912: 'creatinine',
                             51006: 'bun',
                             50971: 'potassium',
                             50902: 'chloride',
                             50893: 'calcium',
                             51222: 'hemoglobin',
                             'Class': 'class'})


In [516]:
data.head()

,subject_id,albumin,bicarbonate,calcium,chloride,creatinine,potassium,bun,hemoglobin,class
0,17158427,4.9,25.0,9.7,94.0,2.1,4.8,77.0,12.7,1
1,18652627,3.8,16.0,8.0,107.0,2.0,4.0,29.0,11.8,1
2,10041990,4.4,33.0,9.7,99.0,1.0,4.4,18.0,10.9,0
3,10857658,4.3,25.0,8.9,101.0,2.7,5.7,41.0,10.4,1
4,10034049,3.8,29.0,9.2,99.0,0.7,4.2,21.0,9.9,0


In [517]:
data['bun_over_creatinine'] = data['bun']/data['creatinine']
data.head()

,subject_id,albumin,bicarbonate,calcium,chloride,creatinine,potassium,bun,hemoglobin,class,bun_over_creatinine
0,17158427,4.9,25.0,9.7,94.0,2.1,4.8,77.0,12.7,1,36.666667
1,18652627,3.8,16.0,8.0,107.0,2.0,4.0,29.0,11.8,1,14.500000
2,10041990,4.4,33.0,9.7,99.0,1.0,4.4,18.0,10.9,0,18.000000
3,10857658,4.3,25.0,8.9,101.0,2.7,5.7,41.0,10.4,1,15.185185
4,10034049,3.8,29.0,9.2,99.0,0.7,4.2,21.0,9.9,0,30.000000


In [518]:
data.to_csv('../Data/modeling_additional_labs.csv',
           index = False)